In [1]:
##### ML model for downscaling agricultural labor in the US 
# Stage 1: train model on sub-national data
# Stage 2: run model at pixel-scale
# Stage 3: disagregate capital to pixel-scale

# AI disclosure: I used Claude to help me write the dictionary of file names
# Disclaimer: I am still in the process of cleaning the data and have not done a thorough EDA, so there are likely errors

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import rasterio
from rasterio.transform import from_bounds
from rasterio.mask import mask
from shapely.geometry import box
import geopandas as gpd
from rasterio.transform import xy

In [2]:
# Get the current working directory
cd = Path.cwd().parent 

# Stage 1: train model on sub-national data

### Step 1: model set-up 

In [5]:
### prepare data

# load data
model_data = pd.read_csv(f"{cd}/Data/Misc./COMPS/USA_LABOR_MASTER.csv")

# set target and predictors 
target = 'log_labor_intensity_jobs_per_million_USD'

predictors = [
    'pct_cropland_irrigated', 'average_travel_time_city',
    'average_travel_time_port', 'probability_economic_land_use_objective',
    'probability_survival_land_use_objective', 'GDP_pc',
    'USD_production_per_HA', 'cereals_share_production_USD',
    'fibres_share_production_USD', 'oilcrops_share_production_USD',
    'pulses_share_production_USD', 'roots_tubers_share_production_USD',
    'rest_of_crops_share_production_USD', 'sugar_crops_share_production_USD',
    'vegetables_share_production_USD', 'rubber_share_production_USD',
    'ruminants_share_production_USD', 'monogastrics_share_production_USD',
    'poultry_share_production_USD', 'share_vlarge_field',
    'share_large_field', 'share_small_field', 'share_vsmall_field'
]

In [6]:
### Set spatial blocks for CV
# 10 blocks = 5 states per block, for a 80/20 split 

n_folds = 10
states = model_data['STATE_ID'].unique()

np.random.seed(31)
np.random.shuffle(states)

state_folds = np.array_split(states, n_folds)  

In [7]:
### Parameters for grid-search 
grid_search = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': ['sqrt', 0.5, 0.75],
}

### Step 2: train model

In [8]:
# set lists to store results
metrics = []
feature_importances = []
models = []

# run model for each fold, including grid search for each 
for fold_idx, test_states in enumerate(state_folds):
    print(f"Fold {fold_idx + 1}/{n_folds}")
    
    # split train and test for fold
    train_mask = ~model_data['STATE_ID'].isin(test_states)
    test_mask  =  model_data['STATE_ID'].isin(test_states)

    X_train = model_data.loc[train_mask, predictors]
    y_train = model_data.loc[train_mask, target]
    X_test  = model_data.loc[test_mask,  predictors]
    y_test  = model_data.loc[test_mask,  target]

    # train model - doing grid search for each fold
    # uses CV within each train dataset to determine 'best' model
    rf = RandomForestRegressor(random_state=27, n_jobs=-1)

    search = RandomizedSearchCV(
        rf, grid_search,
        n_iter=20,
        cv=5,
        scoring='neg_root_mean_squared_error',
        random_state=17,
        n_jobs=-1
    )

    search.fit(X_train, y_train)
    best_model = search.best_estimator_

    # evaluate models based on test data
    y_train_pred = best_model.predict(X_train)
    y_test_pred = best_model.predict(X_test)
    train_r2     = r2_score(y_train, y_train_pred)
    test_r2      = r2_score(y_test,  y_test_pred)

    metrics.append({
        'fold':       fold_idx + 1,
        'best_params': search.best_params_,
        'RMSE':       np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'MAE':        mean_absolute_error(y_test, y_test_pred),
        'R2':         r2_score(y_test, y_test_pred),
        'n_train':    len(y_train),
        'n_test':     len(y_test),
    })

    feature_importances.append(best_model.feature_importances_)
    models.append(best_model)
    
    print(f"  Train R²: {train_r2:.4f}  Test R²: {test_r2:.4f}")

Fold 1/10
  Train R²: 0.9704  Test R²: 0.6049
Fold 2/10
  Train R²: 0.9514  Test R²: 0.6295
Fold 3/10
  Train R²: 0.9684  Test R²: 0.6929
Fold 4/10
  Train R²: 0.9704  Test R²: 0.5721
Fold 5/10
  Train R²: 0.9682  Test R²: 0.7236
Fold 6/10
  Train R²: 0.9688  Test R²: 0.7588
Fold 7/10
  Train R²: 0.9498  Test R²: 0.7874
Fold 8/10
  Train R²: 0.9695  Test R²: 0.6177
Fold 9/10
  Train R²: 0.9500  Test R²: 0.7002
Fold 10/10
  Train R²: 0.9665  Test R²: 0.7040


# Stage 2: run model at pixel-scale

### Step 1: model set-up

In [9]:
#### set filenames for predictor data

rasters = f"{cd}/Data/Misc./COMPS/Predictors/Raster"

precictor_raster_match = {
    'pct_cropland_irrigated':                    'pct_cropland_irrigated.tif',
    'average_travel_time_city':                  'travel_time_city.tif',
    'average_travel_time_port':                  'travel_time_port.tif',
    'probability_economic_land_use_objective':   'economic_objective_prob.tif',
    'probability_survival_land_use_objective':   'survival_objective_prob.tif',
    'GDP_pc':                                    'GDP_per_capita.tif',
    'USD_production_per_HA':                     'USD_production_per_HA.tif',
    'cereals_share_production_USD':              'cereals_share_production_USD.tif',
    'fibres_share_production_USD':               'fibres_share_production_USD.tif',
    'oilcrops_share_production_USD':             'oilcrops_share_production_USD.tif',
    'pulses_share_production_USD':               'pulses_share_production_USD.tif',
    'roots_tubers_share_production_USD':         'roots_tubers_share_production_USD.tif',
    'rest_of_crops_share_production_USD':        'rest_of_crops_share_production_USD.tif',
    'sugar_crops_share_production_USD':          'sugar_crops_share_production_USD.tif',
    'vegetables_share_production_USD':           'vegetables_share_production_USD.tif',
    'rubber_share_production_USD':               'rubber_share_production_USD.tif',
    'ruminants_share_production_USD':            'ruminants_share_production_USD.tif',
    'monogastrics_share_production_USD':         'monogastrics_share_production_USD.tif',
    'poultry_share_production_USD':              'poultry_share_production_USD.tif',
    'share_vlarge_field':                        'field_size_share_vlarge.tif',
    'share_large_field':                         'field_size_share_large.tif',
    'share_small_field':                         'field_size_share_small.tif',
    'share_vsmall_field':                        'field_size_share_vsmall.tif',
}

In [10]:
#### prep predictors (into matrix)

# Use first raster to get spatial reference
first_path = f"{rasters}/{list(precictor_raster_match.values())[0]}"
with rasterio.open(first_path) as src:
    profile  = src.profile.copy()
    nodata   = src.nodata
    shape    = src.shape  

bands = []
for predictor in predictors: 
    path = f"{rasters}/{precictor_raster_match[predictor]}"
    with rasterio.open(path) as src:
        bands.append(src.read(1))  

# stack predictor rasters and flatten into 2D matrix
stack = np.stack(bands, axis=-1)              
orig_shape = stack.shape[:2]
predictor_matrix = stack.reshape(-1, len(predictors))   


# mask no data (NB: I need to do some data filling and trouble shooting on the underlying rasters 
# so this is filter pixels that shouldn't be filtered)
if nodata is not None:
    valid_mask = ~np.any(predictor_matrix == nodata, axis=1)
else:
    valid_mask = ~np.any(np.isnan(predictor_matrix), axis=1)

In [11]:
### Get best model 
best_fold_idx = max(range(len(metrics)), key=lambda i: metrics[i]['R2'])
best_model = models[best_fold_idx]

### Step 2: predict

In [12]:
# run predictions (output is in log scale)
log_predictions = np.full(predictor_matrix.shape[0], np.nan)
log_predictions[valid_mask] = best_model.predict(predictor_matrix[valid_mask])

# undo log 
predictions = np.exp(log_predictions)

# shape back into raster and save
output_raster = predictions.reshape(orig_shape)

output_path = f"{cd}/Data/Misc./COMPS/labor_intensity_predicted.tif"
profile.update(dtype='float32', count=1, nodata=np.nan)

with rasterio.open(output_path, 'w', **profile) as dst:
    dst.write(output_raster.astype('float32'), 1)

/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


# Stage 3: disagregate capital to pixel-scale

### Step 1: calculate prior (intensity x production)

In [13]:
### NB: error here because I've clipped rasters to just continental US for ease, so overestimating by AK and HI

production = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif" 

conus_bbox = box(-125, 24, -66, 50)

with rasterio.open(output_path) as src:
    labor_intensity = src.read(1).astype('float32')
    profile = src.profile.copy()

with rasterio.open(production) as prod_src:
    usd_production, _ = mask(prod_src, [conus_bbox], crop=True) # clipping to continental US to match intensities 
    usd_production = usd_production[0].astype('float32')

labor = labor_intensity * (usd_production / 1e6) # (jobs / mUSD production) * mUSD prodution = jobs

output_path = f"{cd}/Data/Misc./COMPS/prior_labor.tif"
with rasterio.open(output_path, 'w', **profile) as dst:
    dst.write(labor, 1)

### Step 2: rescale to sub-national totals

In [14]:
#### Load and prep sub-national data

# capital stock
sub_national_labor = pd.read_csv(f"{cd}/Data/Misc./COMPS/USA_subnational_labor_intensities.csv")

# geographies
subnational_geo = gpd.read_file(f"{cd}/Data/Misc./COMPS/USA_subnational_geography.shp")

# merge 
subnational_geo = subnational_geo.merge(sub_national_labor, on='PROJ_ID', how='left')

# match raster crs 
subnational_geo = subnational_geo.to_crs(profile['crs'])

In [16]:
#### Assign pixels to counties using their centroids 

# reopen capital estimate and treat no data values as 0's
with rasterio.open(output_path) as src:
    labor = src.read(1).astype('float64')
    profile = src.profile.copy()
    transform = src.transform
    shape = src.shape

nodata_mask = np.isnan(labor)
labor[nodata_mask] = 0.0

# get centroid of each pixel 
rows, cols = np.meshgrid(np.arange(shape[0]), np.arange(shape[1]), indexing='ij')
xs, ys = xy(transform, rows.ravel(), cols.ravel())

pixels_gdf = gpd.GeoDataFrame(
    {'pixel_idx': np.arange(len(xs)),
     'value':     labor.ravel()},
    geometry=gpd.points_from_xy(xs, ys),
    crs=profile['crs']
)

# join pixels to counties based on which one their centroid is in
pixels_gdf = gpd.sjoin(pixels_gdf, subnational_geo[['PROJ_ID', 'ag_jobs_y', 'geometry']], how='left', predicate='within')

In [17]:
#### Rescale pixels so sum matches county total

output = np.full(labor.ravel().shape, np.nan)

for proj_id, group in pixels_gdf.groupby('PROJ_ID'): # for each county
    reported_total = group['ag_jobs_y'].iloc[0] # get county total
    idx = group['pixel_idx'].values

    if pd.isna(reported_total):     # if county reported total is nan, all pixels get nan
        output[idx] = np.nan
    elif reported_total == 0:       # if county reported total is 0, all pixels get 0
        output[idx] = 0.0
    elif group['value'].sum() == 0: # if all pixels add to 0, they all stay 0
        output[idx] = 0.0
    else:                           # otherwise, re-scale so sum matches reported total
        scale = reported_total / group['value'].sum() 
        output[idx] = group['value'].values * scale

output = output.reshape(shape)

# save
profile.update(dtype='float32', nodata=np.nan)
path = f"{cd}/Data/Misc./COMPS/labor_rescaled.tif"

with rasterio.open(path, 'w', **profile) as dst:
    dst.write(output.astype('float32'), 1)